# Import Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


# Load Dataset

In [ ]:
data = pd.read_csv('BsalBdDataFull.csv')

data.columns


# Understanding the Dataset

I wanted to have a list of all the column names, plus definitions as needed, as some of the column names aren't clear initially. The data is explained in the BD_Bsal_Data_Release.xml file.


SwabID - Barcode or submitter-assigned identification number

CollectionDate - Date of field swab collection

TaxaID - Numeric code for species corresponding to the Integrated Taxonomic Information System (ITIS) found at https://www.itis.gov/

ScientificName - Species Name

SiteName - Submitter-assigned site name where swabs were collected

State 

County

Latitude

Longitude

Elevation_m

Sex - Can be Male, Female, or Unknown

Age - Written as NA if age was not ascertained. Otherwise Adult, Juvenile, or Larval.

BodyLength - The body length of the swabbed individual. For amphibians, this is the straight-line distance from the tip of the snout to the vent. It does not include the tail length. An NA is denoted for those records where body length was not measured.

CaptureEnvironment - The habitat in which the swabbed animal was captured. Either aquatic or terrestrial

Bsal.Result - binary result for Bsal PCR, positive = detected, negative = not detected. Bsal is salamander chytrid fungus, which was not detected. This is separate from the result for amphibian chytrid fungus, which was detected. 

Case - Internal lab identifier for group of samples received in the same shipment

Acc - Internal lab identifier for individual sample within a Case

Bdct - Cycle threshold value from PCR reaction (max=40). Lower values indicate more target DNA in sample.  Measured at the PCR cycle where a signal was detected at 5% of max fluorescence and subsequent signal followed a true amplification curve. NA = no signal detected; No Amp = signal detected, but due to signal noise and not amplification of the target; equivocal = signal detected during last cycle with likely amplification, but reaction cycles stopped before true amplification could be determined.

Bd.Result - Interpretation of quantitative PCR results. Positive = true amplification signal detected; Negative = no fluorescent signal detected; No Amp = signal detected, but due to signal noise and not amplification of the target. Manually confirmed during lab QA/QC; equivocal = signal detected during last cycle with likely amplification, but reaction cycles stopped before true amplification could be determined.

copyNum - Calculated estimate of the number of target DNA copies per sample

Gesp - Genus and species used to estimate phylogenetic distance

order - taxonomic order

siteID - numeric index that groups samples within 0.2km to a site

siteVID - numeric index for site visits that occurred greater than 5 days apart at each site

reg - NOAA climate region where the site was sampled

KGzone - 3rd level Köppen-Geiger climate zone where the sample was collected

KG5 - 2nd level Köppen-Geiger climate zone where the sample was collected

KG3 - 1st level Köppen-Geiger climate zone where the sample was collected

m1max - maximum daily temperature estimated at the day of sampling

m7max, m10max, m20max, m30max, m40max, m50max, m60max, m70max, m80max, m90max, m100max, m110max, m120max - average maximum daily temperature estimated for the 7, 10, 20, etc days preceding sampling<

m1min - minimum daily temperature estimated at the day of sampling

m7min, m10min, m20min, m30min, m40min, m50min, m60min, m70min, m80min, m90min, m100min, m110min, m120min - average minimum daily temperature estimated for the 7, 10, 20, etc days preceding sampling

t1precip - total precipitation estimated on the day of sampling

t7precip, t10precip, t20precip, t30precip, t40precip, t50precip, t60precip, t70precip, t80precip, t90precip, t100precip, t110precip, t120precip - total precipitation estimated over the 7, 10, 20, etc days preceding sampling

# Cleaning the Dataset

Columns that should be removed:

SwabID (it is just a sample identifier), TaxaID (It's like scientific name but less detail), SiteName (too specific), County (too specific), Bsal.Result (all of the results were negative), Case (sample identifier), Acc (sample identifier), Bdct (used to determine the value in the Bd column), copyNum (used to determine the value in the Bd column), Gesp (it's the same as scientific name), siteID (we already have values for the site), siteVID (we already have values for the site), KG5 (similar to KGzone), KG3 (similar to KGzone).

Columns that should be changed:

I should separate out Gesp to separate Genus and Species Columns and remove Gesp. For CollectionDate, I will separate out the month into one column, and the year into another column and remove CollectionDate. 

In [ ]:
#Drop unnecessary columns
data = data.drop(columns = ["SwabID","TaxaID", "SiteName", "County", "Bsal.Result", "Case", "Acc", "Bdct", "copyNum", "Gesp", "siteID", "siteVID", "KG5", "KG3"])

#separate the ScientificName column into Genus and Species Columns. Remove ScientificName. 
data[['Genus', 'Species']] = data['ScientificName'].str.split('', n=1, expand = True)
data = data.drop(columns = ['ScientificName'])

#separate the CollectionDate column into Month and Year Columns. Remove CollectionDate
data[['Month', 'Day', 'Year']] = data['CollectionDate'].str.split('/', n = 2, expand = True)
data = data.drop(columns = ['CollectionDate', 'Day'])

I also want to look at columns with unique values or columns with missing values, to see if any more columns or rows should be removed. From this, I can tell that I should drop the Genus column because there's only one Genus. I also want to remove any rows without an elevation, because I think that could be a useful piece of information to study. I should also remove rows without a value in Bd.Result.

In [ ]:
pd.set_option('display.max_rows', None) #tell python to display all the rows

print("Unique values:")
print(data.nunique().sort_values())

print("Missing values:")
print(data.isna().sum().sort_values(ascending=False))

pd.reset_option('display.max_rows') #tell python to stop displaying all the rows

In [ ]:
#Remove unnecessary columns
data = data.drop(columns = ["Genus"])

#Remove any rows with NA
data = data.dropna(subset=['Elevation_m'])
data = data.dropna(subset=['Bd.Result'])
data = data.dropna(subset=['BodyLength'])
data = data.dropna(subset=['Sex'])
data = data.dropna(subset=['Age'])
data = data.dropna(subset=['Species'])

# Data Preparation

I need to make sure all of my data types are correct

In [ ]:
#change month and year to numeric
data['Month'] = pd.to_numeric(data['Month'])
data['Year'] = pd.to_numeric(data['Year'])

#Convert the values of Bd.Result to 0 or 1
data['Bd.Result'] = data['Bd.Result'].map({'positive': 1,'negative': 0})


#for logistic regression, I can't use strings, I need numbers
data = pd.get_dummies(data, columns=['C'])

print(data.dtypes)

I need to separate my data features (x) from my results (y). I also need to create training and test data. 

In [ ]:
#separate my features from my result
x = data.drop(columns = "Bd.Result")
y = data['Bd.Result']

#create training and test data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 1)


# Start Testing Models

To start, I wanted to just try something basic, so I used logistic regression

In [ ]:
LogReg = LogisticRegression(random_state = 1, max_iter = 10000)
LogReg.fit(x_train, y_train)

